# ENGR 422 — Assignment 2: Camera Calibration and Depth Estimation
**Student:** Mohammed Mahdi  
**Date:** April 2026

---
## Overview
This notebook implements a complete stereo vision pipeline:
1. Camera calibration using a checkerboard pattern
2. Stereo image rectification
3. Manual depth estimation using the stereo depth equation
4. Dense disparity map generation
5. Error analysis and discussion


## 1. Setup and Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")


## 2. Configuration

**Checkerboard pattern:** 10 × 7 inner corners (11 × 8 squares)  
**Square size:** 25 mm  
**Baseline:** 80 mm (distance between left and right camera positions)

### Ground Truth Measurements (measured with ruler/tape)
| Object | Distance from Camera |
|--------|---------------------|
| Red Box (nearest) | 350 mm |
| Pringles Can (middle) | 650 mm |
| Hedgehog Figurines (farthest) | 950 mm |


In [ ]:
# Checkerboard parameters
CHECKERBOARD = (10, 7)   # inner corners (columns, rows)
SQUARE_SIZE  = 25.0      # mm

# Stereo parameters
BASELINE = 80.0  # mm

# Ground truth distances (camera to object, in mm)
GROUND_TRUTH = {
    'Red Box':              350.0,
    'Pringles Can':         650.0,
    'Hedgehog Figurines':   950.0,
}

# Processing scale (half resolution for speed + better reprojection error)
SCALE = 0.5

# Paths
CALIB_DIR  = 'images/calibration'
STEREO_DIR = 'images/stereo'
SETUP_DIR  = 'images/setup'


## 3. Load Calibration Images

We captured **15 images** of a checkerboard pattern from various angles and positions. 
The images were taken with an iPhone camera in portrait orientation (4284 × 5712 px).

Since the stereo images are in **landscape** orientation, we rotate all calibration images 
90° clockwise to ensure consistent camera intrinsics.


In [ ]:
# Load calibration images, rotate to landscape, and resize
calib_files = sorted(glob.glob(os.path.join(CALIB_DIR, 'calib_*.jpg')))
print(f"Found {len(calib_files)} calibration images")

calib_images = []
for f in calib_files:
    img = cv2.imread(f)
    # Rotate 90° clockwise: portrait → landscape
    img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    # Resize
    img = cv2.resize(img, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)
    calib_images.append(img)
    
h, w = calib_images[0].shape[:2]
print(f"Processed size: {w} × {h} pixels (landscape)")

# Show a sample grid of 6 calibration images
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(calib_images):
        ax.imshow(cv2.cvtColor(calib_images[i], cv2.COLOR_BGR2RGB))
        ax.set_title(f'Calibration Image {i+1}', fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Calibration Images (Rotated to Landscape)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Checkerboard Corner Detection

We use `cv2.findChessboardCorners()` to detect the 10 × 7 = 70 inner corners in each image.
Corners are then refined to sub-pixel accuracy using `cv2.cornerSubPix()`.


In [ ]:
# Termination criteria for corner sub-pixel refinement
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Prepare 3D object points: (0,0,0), (25,0,0), (50,0,0), ... in mm
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE

objpoints = []  # 3D points in real world
imgpoints = []  # 2D points in image plane

successful = []
failed = []

for i, img in enumerate(calib_images):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    flags = (cv2.CALIB_CB_ADAPTIVE_THRESH + 
             cv2.CALIB_CB_FAST_CHECK + 
             cv2.CALIB_CB_NORMALIZE_IMAGE)
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, flags)
    
    if ret:
        corners_refined = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        objpoints.append(objp)
        imgpoints.append(corners_refined)
        successful.append(i)
    else:
        failed.append(i)

print(f"Corner detection: {len(successful)}/{len(calib_images)} successful")
if failed:
    print(f"Failed images: {[f+1 for f in failed]}")


In [ ]:
# Visualize detected corners on a few images
n_show = min(6, len(successful))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, ax in enumerate(axes.flat):
    if idx < n_show:
        img_idx = successful[idx]
        img_vis = calib_images[img_idx].copy()
        gray = cv2.cvtColor(img_vis, cv2.COLOR_BGR2GRAY)
        ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD,
            cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE)
        if ret:
            corners = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
            cv2.drawChessboardCorners(img_vis, CHECKERBOARD, corners, ret)
        ax.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Image {img_idx+1}', fontsize=10)
    ax.axis('off')

plt.suptitle('Detected Checkerboard Corners', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Camera Calibration

We use `cv2.calibrateCamera()` to compute:
- **Camera matrix (K):** Contains focal lengths (fx, fy) and principal point (cx, cy)
- **Distortion coefficients:** Models lens distortion (radial and tangential)
- **Rotation and translation vectors:** Extrinsic parameters for each calibration image


In [ ]:
# Run camera calibration
ret, K, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, (w, h), None, None
)

print("=" * 60)
print("CAMERA CALIBRATION RESULTS")
print("=" * 60)
print(f"\nReprojection Error (RMS): {ret:.4f} pixels")
print(f"  → This measures calibration accuracy.")
print(f"  → Values < 1.0 px are excellent, < 2.0 px are good.")
print(f"\n{'─' * 60}")
print(f"Camera Matrix (K):")
print(f"  ┌ {K[0,0]:10.2f}  {K[0,1]:10.2f}  {K[0,2]:10.2f} ┐")
print(f"  │ {K[1,0]:10.2f}  {K[1,1]:10.2f}  {K[1,2]:10.2f} │")
print(f"  └ {K[2,0]:10.2f}  {K[2,1]:10.2f}  {K[2,2]:10.2f} ┘")


## 6. Intrinsic Parameter Interpretation

The camera matrix **K** has the form:

$$
K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}
$$

Each parameter has a clear physical meaning explained below.


In [ ]:
fx, fy = K[0, 0], K[1, 1]
cx, cy = K[0, 2], K[1, 2]

print("INTRINSIC PARAMETERS — Physical Interpretation")
print("=" * 60)
print(f"\n  fx = {fx:.2f} pixels")
print(f"  fy = {fy:.2f} pixels")
print(f"  → Focal length in pixels along X and Y axes.")
print(f"  → These convert real-world angles to pixel distances.")
print(f"  → fx ≈ fy indicates square pixels (expected for modern cameras).")

print(f"\n  cx = {cx:.2f} pixels")
print(f"  cy = {cy:.2f} pixels")
print(f"  → Principal point: where the optical axis hits the sensor.")
print(f"  → Ideally near the image center ({w/2:.0f}, {h/2:.0f}).")
print(f"  → Offset of ({abs(cx - w/2):.1f}, {abs(cy - h/2):.1f}) px from center is normal.")

print(f"\n{'─' * 60}")
print(f"DISTORTION COEFFICIENTS")
print(f"  k1 = {dist_coeffs[0,0]:+.6f}  (radial, 2nd order)")
print(f"  k2 = {dist_coeffs[0,1]:+.6f}  (radial, 4th order)")
print(f"  p1 = {dist_coeffs[0,2]:+.6f}  (tangential)")
print(f"  p2 = {dist_coeffs[0,3]:+.6f}  (tangential)")
print(f"  k3 = {dist_coeffs[0,4]:+.6f}  (radial, 6th order)")
print(f"\n  → Radial distortion (k1, k2, k3): causes barrel/pincushion effect.")
print(f"  → Tangential distortion (p1, p2): caused by lens not perfectly parallel to sensor.")
print(f"  → k1 > 0 suggests slight pincushion distortion.")


## 7. Per-Image Reprojection Error

In [ ]:
# Calculate per-image reprojection error
errors = []
for i in range(len(objpoints)):
    projected, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, dist_coeffs)
    error = cv2.norm(imgpoints[i], projected, cv2.NORM_L2) / len(projected)
    errors.append(error)

plt.figure(figsize=(12, 5))
bars = plt.bar(range(1, len(errors)+1), errors, color='steelblue', edgecolor='navy', alpha=0.8)
plt.axhline(y=ret, color='red', linestyle='--', linewidth=2, label=f'Mean RMS = {ret:.4f} px')
plt.xlabel('Calibration Image', fontsize=12)
plt.ylabel('Reprojection Error (pixels)', fontsize=12)
plt.title('Per-Image Reprojection Error', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.xticks(range(1, len(errors)+1))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBest image:  #{np.argmin(errors)+1} (error = {min(errors):.4f} px)")
print(f"Worst image: #{np.argmax(errors)+1} (error = {max(errors):.4f} px)")


## 8. Stereo Rectification (Undistortion)

Using the calibration matrix $K$ and distortion coefficients, we undistort the left and right stereo images.

Since our camera moved precisely parallel along a straight edge on the table (with no rotation), the setup acts as an **ideal parallel stereo rig**. Thus, we only need to correct for lens distortion using `cv2.undistort()`.


In [ ]:
# Load stereo images
left_img = cv2.imread(os.path.join(STEREO_DIR, 'left.jpg'))
right_img = cv2.imread(os.path.join(STEREO_DIR, 'right.jpg'))

# Verify they exist
if left_img is None or right_img is None:
    raise ValueError("Could not find stereo images!")

# Resize to match calibration scale
left_img = cv2.resize(left_img, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)
right_img = cv2.resize(right_img, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)

# Get optimal new camera matrix (removes black borders caused by undistortion)
new_K, roi = cv2.getOptimalNewCameraMatrix(K, dist_coeffs, (w,h), 1, (w,h))

# Undistort both images
left_rect = cv2.undistort(left_img, K, dist_coeffs, None, new_K)
right_rect = cv2.undistort(right_img, K, dist_coeffs, None, new_K)

# Crop to ROI
x, y, w_roi, h_roi = roi
left_rect = left_rect[y:y+h_roi, x:x+w_roi]
right_rect = right_rect[y:y+h_roi, x:x+w_roi]

print(f"Original shape: {left_img.shape}")
print(f"Rectified shape: {left_rect.shape}")

# Visualize Rectification
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].imshow(cv2.cvtColor(left_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Left Image (Distorted)", fontsize=12)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(left_rect, cv2.COLOR_BGR2RGB))
axes[1].set_title("Rectified Left Image (Undistorted & Cropped)", fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()


## 9. Feature Matching & Manual Depth Calculation

The stereo depth equation relates disparity ($d$) to real-world depth ($Z$):
$$ Z = \frac{f \cdot b}{d} $$
Where:
- $Z$ is depth (mm)
- $f$ is focal length (pixels)
- $b$ is the baseline between cameras (80 mm)
- $d = x_L - x_R$ is the disparity (difference in horizontal pixel coordinate)

We will use **SIFT (Scale-Invariant Feature Transform)** to automatically find the exact pixel coordinates ($x_L, y_L$) and ($x_R, y_R$) of points on our three target objects.


In [ ]:
# Find correspondences using SIFT
sift = cv2.SIFT_create()
kp_left, des_left = sift.detectAndCompute(left_rect, None)
kp_right, des_right = sift.detectAndCompute(right_rect, None)

# FLANN matcher
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)

matches = flann.knnMatch(des_left, des_right, k=2)

# Apply Lowe's ratio test to filter good matches
good_matches = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

print(f"Found {len(good_matches)} good feature matches across the images.")


### 9.1. Object Correspondences
We extract points belonging to each object to calculate depth manually.

In [ ]:
# Define rough bounding boxes for the 3 objects to filter matches automatically
# Format: (x_min, x_max, y_min, y_max)
regions = {
    'Red Box':              (100,  800,  1500, 2500), # Approximate region in left image
    'Pringles Can':         (1300, 1600, 1100, 1800),
    'Hedgehog Figurines':   (1800, 2400, 1500, 1800)
}

selected_points = {}
fx_new = new_K[0,0]  # Use focal length from the new optimal matrix

print("MANUAL DEPTH CALCULATION EXAMPLES")
print("=" * 65)
print(f"Formula: Z = (fx * baseline) / (x_L - x_R)")
print(f"fx = {fx_new:.2f} px")
print(f"Baseline = {BASELINE} mm\n")

for obj_name, box in regions.items():
    xmin, xmax, ymin, ymax = box
    
    # Find a good match inside this box
    match_found = None
    for m in good_matches:
        pt_L = kp_left[m.queryIdx].pt
        if xmin < pt_L[0] < xmax and ymin < pt_L[1] < ymax:
            # Check epipolar constraint (y_L should be very close to y_R)
            pt_R = kp_right[m.trainIdx].pt
            if abs(pt_L[1] - pt_R[1]) < 10.0:  # Allow max 10px vertical difference
                match_found = m
                break
                
    if match_found:
        pt_L = kp_left[match_found.queryIdx].pt
        pt_R = kp_right[match_found.trainIdx].pt
        
        x_L, y_L = pt_L
        x_R, y_R = pt_R
        disparity = abs(x_L - x_R)
        
        # Calculate depth
        # Z = f * B / d
        calculated_depth = (fx_new * BASELINE) / disparity
        
        selected_points[obj_name] = {
            'x_L': x_L, 'x_R': x_R, 
            'y_L': y_L, 'y_R': y_R,
            'd': disparity,
            'calc_z': calculated_depth
        }
        
        print(f"► {obj_name}:")
        print(f"    Left pixel (xL, yL):  ({x_L:6.1f}, {y_L:6.1f})")
        print(f"    Right pixel (xR, yR): ({x_R:6.1f}, {y_R:6.1f})")
        print(f"    Disparity (d):        {disparity:6.1f} px")
        print(f"    Depth (Z):            ({fx_new:.1f} * {BASELINE}) / {disparity:.1f} = {calculated_depth:.1f} mm")
        print("-" * 65)


In [ ]:
# Visualize the selected points on the images
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
img_L_pts = left_rect.copy()
img_R_pts = right_rect.copy()

colors = [(0, 0, 255), (0, 255, 0), (255, 0, 0)] # Red, Green, Blue
for idx, (name, data) in enumerate(selected_points.items()):
    c = colors[idx]
    
    # Left
    cv2.circle(img_L_pts, (int(data['x_L']), int(data['y_L'])), 15, c, -1)
    cv2.putText(img_L_pts, name, (int(data['x_L'])-50, int(data['y_L'])-25), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, c, 3)
    
    # Right
    cv2.circle(img_R_pts, (int(data['x_R']), int(data['y_R'])), 15, c, -1)
    cv2.putText(img_R_pts, name, (int(data['x_R'])-50, int(data['y_R'])-25), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, c, 3)

axes[0].imshow(cv2.cvtColor(img_L_pts, cv2.COLOR_BGR2RGB))
axes[0].set_title("Left Image Correspondences", fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_R_pts, cv2.COLOR_BGR2RGB))
axes[1].set_title("Right Image Correspondences", fontsize=14, fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()


## 10. Depth Comparison & Error Analysis

The assignment rubric requires a 7-column table comparing measured and calculated depth.

**Error Equations:**
- Absolute Error = $| Calculated - Measured |$
- Percentage Error = $(Absolute Error / Measured) \times 100\%$


In [ ]:
import pandas as pd

table_data = []

for obj_name, data in selected_points.items():
    measured_Z = GROUND_TRUTH[obj_name]
    calc_Z = data['calc_z']
    
    abs_err = abs(calc_Z - measured_Z)
    pct_err = (abs_err / measured_Z) * 100
    
    table_data.append({
        'Object': obj_name,
        'Pixel L (xL, yL)': f"({data['x_L']:.1f}, {data['y_L']:.1f})",
        'Pixel R (xR, yR)': f"({data['x_R']:.1f}, {data['y_R']:.1f})",
        'Disparity (d) px': f"{data['d']:.1f}",
        'Measured Depth (Z) mm': f"{measured_Z:.1f}",
        'Calculated Depth (Z) mm': f"{calc_Z:.1f}",
        'Abs Error (mm)': f"{abs_err:.1f}",
        'Error (%)': f"{pct_err:.2f}%"
    })

df = pd.DataFrame(table_data)

# Print nice formatted table
print("\n" + "=" * 130)
print(f"{'DEPTH ESTIMATION COMPARISON TABLE':^130}")
print("=" * 130)
print(df.to_string(index=False))
print("=" * 130)


## 11. Dense Disparity Map

We generate a dense depth map for the entire scene using `StereoSGBM` (Semi-Global Block Matching).


In [ ]:
# Convert to grayscale for stereo matching
gray_L = cv2.cvtColor(left_rect, cv2.COLOR_BGR2GRAY)
gray_R = cv2.cvtColor(right_rect, cv2.COLOR_BGR2GRAY)

# Downscale to 25% (This is the secret to removing the noise for high-res images!)
stereo_scale = 0.25
sgbm_L = cv2.resize(gray_L, None, fx=stereo_scale, fy=stereo_scale)
sgbm_R = cv2.resize(gray_R, None, fx=stereo_scale, fy=stereo_scale)

# Configure StereoSGBM with a much larger block size
window_size = 15  # Larger window sees more texture = less dots
min_disp = 0
num_disp = 160    # Must be divisible by 16

stereo = cv2.StereoSGBM_create(
    minDisparity=min_disp,
    numDisparities=num_disp,
    blockSize=window_size,
    P1=8 * 3 * window_size**2,
    P2=32 * 3 * window_size**2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=2,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)

# Compute disparity map
disp = stereo.compute(sgbm_L, sgbm_R).astype(np.float32) / 16.0

# Filter out invalid pixels
disp[disp <= min_disp] = np.nan

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(22, 10))

# Show Original Image (Left Rectified)
axes[0].imshow(cv2.cvtColor(left_rect, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Scene (Left Image)', fontsize=16, fontweight='bold')
axes[0].axis('off')

# Show Disparity Map
im = axes[1].imshow(disp, cmap='jet')
cbar = plt.colorbar(im, ax=axes[1], shrink=0.7)
cbar.set_label('Disparity (pixels)', fontsize=12)
axes[1].set_title('Dense Disparity Map', fontsize=16, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()


## 12. Discussion

*(Answers to the rubric questions based on our results)*

### Q1: Is depth estimation less accurate for distant objects? Why?
**Yes.** Looking at our comparison table, the absolute error for the Hedgehog Figurines (farthest object at 950mm) is likely higher than the Red Box (350mm). 
Because $Z = fb/d$, depth ($Z$) is inversely proportional to disparity ($d$). As distance increases, disparity becomes very small. A 1-pixel matching error on a far object (small $d$) causes a massive jump in calculated depth, whereas a 1-pixel error on a close object (large $d$) has a very minor effect.

### Q2: What was the primary source of error in your experiment?
1. **Correspondence Uncertainty:** In our manual feature matching, selecting the *exact* same 3D point in both images is subject to ±1-2 pixel error. Due to the inverse relationship, even 2 pixels of error at $d=50$px causes significant depth shift.
2. **Setup Geometry:** Our baseline measurement was manually done with a ruler (80 mm). If the true baseline was 82 mm, the entire calculation is scaled off by 2.5%.
3. **Camera Alignment:** We slid the phone along the table edge. Any slight rotation or pitch introduced minor vertical disparities (epipolar misalignment) not completely fixed by radial undistortion.

### Q3: How would increasing the baseline affect the results?
Increasing the baseline ($b$) directly increases the disparity ($d$) for all objects, since $d = (fb)/Z$.
- **Pro:** A larger disparity reduces the percentage impact of a 1-pixel matching error, significantly improving depth resolution for *far* objects.
- **Con:** It increases the difference in viewing angles, causing severe *occlusions* (where an object is visible to one camera but blocked by another object in the second camera), making correspondence matching impossible for those regions.

### Q4: Why is depth estimation difficult for low-texture or highly reflective objects?
- **Low Texture (e.g. flat walls):** SGBM and SIFT algorithms rely on comparing pixel intensity gradients (corners, edges). A blank white wall looks identical everywhere, so the algorithm cannot find unique matching points, causing "holes" in the disparity map.
- **Reflective Objects:** Specular highlights (reflections) change position depending on the camera angle. A bright spot on a bottle will appear at different physical coordinates in the left and right images, leading the stereo algorithm to calculate false disparities.

### Q5: How would you improve this setup for an autonomous robot?
1. **Hardware Synchronisation:** Use a dedicated stereo camera rig (like Intel RealSense) with fixed, factory-calibrated lenses to eliminate manual baseline error and movement inconsistencies.
2. **Global Shutter:** Mobile phones use rolling shutters which distort moving scenes. Global shutters capture all pixels instantly.
3. **Active Stereo (Structured Light):** Project an IR dot pattern onto the scene. This provides artificial "texture" to textureless regions (like walls), solving the issue discussed in Q4 and allowing perfect dense depth mapping.
